In [1]:
from pathlib import Path
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Locate project root
project_root = Path.cwd().resolve()

if not (project_root / "data").is_dir():
    project_root = project_root.parent

data_path = project_root / "data/raw/yellow_tripdata_2025-01.parquet"

assert data_path.exists(), f"Data not found: {data_path}"

con = duckdb.connect()

print("Project root:", project_root)
print("Data:", data_path.name)
print("File size (MB):", round(data_path.stat().st_size / 1024**2, 2))

Project root: /Users/wushixundelaopo/Desktop/Urban_Mobility_Mining
Data: yellow_tripdata_2025-01.parquet
File size (MB): 56.42


In [2]:
# Section 1.1 — Dataset Schema

schema = con.execute(
    "DESCRIBE SELECT * FROM read_parquet(?)",
    [str(data_path)]
).df()

display(schema)

,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,tpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,tpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,passenger_count,BIGINT,YES,None,None,None
4,trip_distance,DOUBLE,YES,None,None,None
5,RatecodeID,BIGINT,YES,None,None,None
6,store_and_fwd_flag,VARCHAR,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
8,DOLocationID,INTEGER,YES,None,None,None
9,payment_type,BIGINT,YES,None,None,None


In [3]:
# Section 1.2 — Basic Dataset Statistics

overview = con.execute("""
    SELECT
        COUNT(*) AS total_trips,
        MIN(tpep_pickup_datetime) AS earliest_pickup,
        MAX(tpep_pickup_datetime) AS latest_pickup,
        COUNT(DISTINCT PULocationID) AS n_pickup_zones,
        COUNT(DISTINCT DOLocationID) AS n_dropoff_zones
    FROM read_parquet(?)
""", [str(data_path)]).df()

display(overview)

,total_trips,earliest_pickup,latest_pickup,n_pickup_zones,n_dropoff_zones
0,3475226,2024-12-31 20:47:55,2025-02-01 00:00:44,261,260


In [4]:
# Section 2.1 — Missing Values and Data Quality

quality = con.execute("""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE tpep_pickup_datetime IS NULL
        ) AS missing_pickup_time,

        COUNT(*) FILTER (
            WHERE tpep_dropoff_datetime IS NULL
        ) AS missing_dropoff_time,

        COUNT(*) FILTER (
            WHERE PULocationID IS NULL
        ) AS missing_pickup_zone,

        COUNT(*) FILTER (
            WHERE DOLocationID IS NULL
        ) AS missing_dropoff_zone,

        COUNT(*) FILTER (
            WHERE trip_distance IS NULL
        ) AS missing_distance,

        COUNT(*) FILTER (
            WHERE tpep_dropoff_datetime < tpep_pickup_datetime
        ) AS negative_duration,

        COUNT(*) FILTER (
            WHERE tpep_dropoff_datetime = tpep_pickup_datetime
        ) AS zero_duration,

        COUNT(*) FILTER (
            WHERE trip_distance < 0
        ) AS negative_distance,

        COUNT(*) FILTER (
            WHERE trip_distance = 0
        ) AS zero_distance,

        COUNT(*) FILTER (
            WHERE PULocationID = DOLocationID
        ) AS same_origin_destination,

        COUNT(*) FILTER (
            WHERE tpep_pickup_datetime < TIMESTAMP '2025-01-01'
               OR tpep_pickup_datetime >= TIMESTAMP '2025-02-01'
        ) AS outside_january

    FROM read_parquet(?)
""", [str(data_path)]).df()

display(quality.T.rename(columns={0: "count"}))

,count
total_rows,3475226
missing_pickup_time,0
missing_dropoff_time,0
missing_pickup_zone,0
missing_dropoff_zone,0
missing_distance,0
negative_duration,124
zero_duration,1927
negative_distance,0
zero_distance,90893


In [5]:
# Section 2.2 — Investigate Zero-Distance Trips

zero_distance_audit = con.execute("""
    SELECT
        COUNT(*) AS zero_distance_trips,

        COUNT(*) FILTER (
            WHERE tpep_pickup_datetime = tpep_dropoff_datetime
        ) AS also_zero_duration,

        COUNT(*) FILTER (
            WHERE tpep_dropoff_datetime < tpep_pickup_datetime
        ) AS also_negative_duration,

        COUNT(*) FILTER (
            WHERE PULocationID = DOLocationID
        ) AS also_same_zone,

        COUNT(*) FILTER (
            WHERE trip_distance = 0
              AND PULocationID <> DOLocationID
        ) AS zero_distance_different_zones

    FROM read_parquet(?)
    WHERE trip_distance = 0
""", [str(data_path)]).df()

display(zero_distance_audit.T.rename(columns={0: "count"}))

,count
zero_distance_trips,90893
also_zero_duration,692
also_negative_duration,0
also_same_zone,30079
zero_distance_different_zones,60814


## Section 3 — Full-Year Dataset Audit (2025)

This section examines all 12 monthly NYC TLC Yellow Taxi Parquet files to assess data coverage, consistency, and quality before defining the shared preprocessing pipeline.

In [6]:
# Section 3.1 — Verify Monthly Parquet Files

monthly_files = sorted(
    (project_root / "data/raw").glob("yellow_tripdata_2025-*.parquet")
)

assert len(monthly_files) == 12, (
    f"Expected 12 monthly files, found {len(monthly_files)}"
)

file_info = []

for file in monthly_files:
    row_count = con.execute(
        "SELECT COUNT(*) FROM read_parquet(?)",
        [str(file)]
    ).fetchone()[0]

    file_info.append({
        "month": file.stem[-2:],
        "filename": file.name,
        "rows": row_count,
        "size_mb": round(file.stat().st_size / 1024**2, 2)
    })

monthly_overview = pd.DataFrame(file_info)

display(monthly_overview)

print("Total trips:", f"{monthly_overview['rows'].sum():,}")
print("Total size (MiB):", round(monthly_overview["size_mb"].sum(), 2))

,month,filename,rows,size_mb
0,01,yellow_tripdata_2025-01.parquet,3475226,56.42
1,02,yellow_tripdata_2025-02.parquet,3577543,57.55
2,03,yellow_tripdata_2025-03.parquet,4145257,66.72
3,04,yellow_tripdata_2025-04.parquet,3970553,64.23
4,05,yellow_tripdata_2025-05.parquet,4591845,74.23
5,06,yellow_tripdata_2025-06.parquet,4322960,70.14
6,07,yellow_tripdata_2025-07.parquet,3898963,63.84
7,08,yellow_tripdata_2025-08.parquet,3574091,59.41
8,09,yellow_tripdata_2025-09.parquet,4251015,69.08
9,10,yellow_tripdata_2025-10.parquet,4428699,71.78


Total trips: 48,722,602
Total size (MiB): 791.53


In [7]:
# Section 3.2 — Monthly Data Quality Audit

monthly_quality = []

for file in monthly_files:
    month = file.stem[-2:]
    start = f"2025-{month}-01"
    end = (
        f"2026-01-01" if month == "12"
        else f"2025-{int(month) + 1:02d}-01"
    )

    result = con.execute("""
        SELECT
            COUNT(*) AS total_rows,

            COUNT(*) FILTER (
                WHERE tpep_pickup_datetime IS NULL
                   OR tpep_dropoff_datetime IS NULL
                   OR PULocationID IS NULL
                   OR DOLocationID IS NULL
            ) AS missing_core_fields,

            COUNT(*) FILTER (
                WHERE tpep_dropoff_datetime < tpep_pickup_datetime
            ) AS negative_duration,

            COUNT(*) FILTER (
                WHERE tpep_dropoff_datetime = tpep_pickup_datetime
            ) AS zero_duration,

            COUNT(*) FILTER (
                WHERE trip_distance = 0
            ) AS zero_distance,

            COUNT(*) FILTER (
                WHERE PULocationID = DOLocationID
            ) AS same_zone,

            COUNT(*) FILTER (
                WHERE tpep_pickup_datetime < CAST(? AS TIMESTAMP)
                   OR tpep_pickup_datetime >= CAST(? AS TIMESTAMP)
            ) AS outside_month

        FROM read_parquet(?)
    """, [start, end, str(file)]).df()

    result.insert(0, "month", month)
    monthly_quality.append(result)

monthly_quality = pd.concat(
    monthly_quality, ignore_index=True
)

for col in [
    "missing_core_fields",
    "negative_duration",
    "zero_duration",
    "zero_distance",
    "same_zone",
    "outside_month"
]:
    monthly_quality[col + "_pct"] = (
        monthly_quality[col] /
        monthly_quality["total_rows"] * 100
    )

display(monthly_quality[
    [
        "month",
        "total_rows",
        "missing_core_fields",
        "negative_duration",
        "zero_distance_pct",
        "same_zone_pct",
        "outside_month"
    ]
].round(3))

,month,total_rows,missing_core_fields,negative_duration,zero_distance_pct,same_zone_pct,outside_month
0,01,3475226,0,124,2.615,5.194,22
1,02,3577543,0,93,2.789,4.906,31
2,03,4145257,0,81,2.502,4.725,33
3,04,3970553,0,163,2.303,4.787,7
4,05,4591845,0,101,3.073,4.573,24
5,06,4322960,0,231,3.120,4.486,20
6,07,3898963,0,1,3.175,4.539,7
7,08,3574091,0,2,2.943,4.457,17
8,09,4251015,0,0,2.928,4.544,7
9,10,4428699,0,2,2.836,4.517,13


In [8]:
# Section 3.3 — Negative Duration Investigation

negative_duration_details = []

for file in monthly_files:
    month = file.stem[-2:]

    result = con.execute("""
        SELECT
            CAST(tpep_pickup_datetime AS DATE) AS pickup_date,
            COUNT(*) AS negative_trips,
            MIN(
                date_diff(
                    'minute',
                    tpep_pickup_datetime,
                    tpep_dropoff_datetime
                )
            ) AS min_duration_minutes,
            MAX(
                date_diff(
                    'minute',
                    tpep_pickup_datetime,
                    tpep_dropoff_datetime
                )
            ) AS max_duration_minutes
        FROM read_parquet(?)
        WHERE tpep_dropoff_datetime < tpep_pickup_datetime
        GROUP BY 1
    """, [str(file)]).df()

    result.insert(0, "file_month", month)
    negative_duration_details.append(result)

negative_duration_details = pd.concat(
    negative_duration_details, ignore_index=True
)

display(
    negative_duration_details.sort_values(
        "negative_trips", ascending=False
    ).head(20)
)

,file_month,pickup_date,negative_trips,min_duration_minutes,max_duration_minutes
171,11,2025-11-02,1431,-58,-1
160,06,2025-06-20,28,0,0
158,06,2025-06-18,20,0,0
159,06,2025-06-19,18,0,0
156,06,2025-06-16,16,0,0
157,06,2025-06-17,16,0,0
151,06,2025-06-11,16,0,0
100,04,2025-04-20,15,-1,0
150,06,2025-06-10,15,0,0
96,04,2025-04-16,13,0,0
